<a href="https://colab.research.google.com/github/YuliiaDina/transformer-survival-competing-risks-bachelor/blob/main/main_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Завантаження бібліотек

In [ ]:
import sys
import os

GITHUB_USER = "YuliiaDina"
REPO_NAME = "transformer-survival-competing-risks-bachelor"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"


if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull origin main
    %cd ..

%cd {REPO_NAME}
PROJECT_PATH = os.path.abspath(".")

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

!pip install -q scikit-survival comprisk
!pip install -q poetry
!poetry config virtualenvs.create false
!poetry install --no-root -q

#Завантаження модулів нашого проекту з GitHub

In [ ]:
from src.data_prep import load_and_prepare_data, split_and_scale_data, create_time_bins, prepare_tensor_loaders, prepare_survival_data
from src.models import FinalTransformerModel
from src.baselines import train_cox_models, train_fine_gray_models
from src.engine import train_model_with_history
from src.metrics import compute_cox_cif
from src.visualization import plot_loss_curves, plot_cif_comparison,plot_calibration_curves_survival
from src.tables import build_table_1_demographics, build_table_2_metrics, build_table_3_hyperparams
from src.utils import run_cif_checks

print("Усі модулі успішно імпортовано")

#Завантаження та обробка данних

In [ ]:
# 1. Завантаження датасету та розбиття на Train/Val/Test із масштабуванням ознак
df = load_and_prepare_data()
features = ['age', 'sex', 'dxyr', 'hgb', 'creat', 'mspike']
df_train, df_val, df_test, X_train_scaled, X_val_scaled, X_test_scaled = split_and_scale_data(df, features)

# 2. Квантилізація часу на 20 інтервалів та створення DataLoader для PyTorch
NUM_BINS = 20
bins = create_time_bins(df_train['time'].values, num_bins=NUM_BINS)
train_loader, val_loader, X_train_tensor, X_val_tensor, X_test_tensor = prepare_tensor_loaders(
    X_train_scaled, df_train, X_val_scaled, df_val, X_test_scaled, df_test, bins, batch_size=64
)

print("\n Дані оброблено, тензори та вибірки сформовано")

#Навчання статичник моделей та транформетрів варіантів А та В

In [ ]:

print("\n Класичні статистичні базові моделі")
models_cox = train_cox_models(df_train, X_train_scaled)
models_fg = train_fine_gray_models(df_train, X_train_scaled)

print("\nTransformers")

# Навчання Варіанту А: Стандартна архітектура
model_A_hist, train_hist_A, val_hist_A = train_model_with_history(
    variant_name='A',
    train_loader=train_loader,
    val_loader=val_loader,
    n_features=6,
    num_bins=NUM_BINS,
    max_epochs=200,
    patience=15
)

# Навчання Варіанту В: Архітектура з монотонною увагою (Monotonic Attention)
model_B_hist, train_hist_B, val_hist_B = train_model_with_history(
    variant_name='B',
    train_loader=train_loader,
    val_loader=val_loader,
    n_features=6,
    num_bins=NUM_BINS,
    max_epochs=200,
    patience=15
)

print("\n Навчання моделей закінчено")

#Валідація данних та графічна візуалізація отриманих результатів

In [ ]:
import numpy as np

# 1. Верифікація математичної коректності функцій інцидентності
run_cif_checks(model_A_hist, 'A (Стандартний)', X_test_tensor)
run_cif_checks(model_B_hist, 'B (Monotonic)', X_test_tensor)

# 2. Графіки збіжності кривих навчання (Loss Curves)
plot_loss_curves(train_hist_A, val_hist_A, train_hist_B, val_hist_B)

# 3. Побудова порівняльних графіків функцій CIF
min_time = np.percentile(df_test['time'], 10)
max_time = np.percentile(df_test['time'], 90)
brier_times = np.linspace(min_time, max_time, 100)

plot_cif_comparison(models_cox, model_A_hist, model_B_hist, X_test_scaled, X_test_tensor, brier_times, bins)

# 4. Побудова графіків калібрування (на горизонт 5 років / 60 місяців)
plot_calibration_curves_survival(
    models_cox,model_A_hist,model_B_hist,X_test_scaled,X_test_tensor,df_test,bins,
    target_time=60.0,max_limits={1: 0.2, 2: 1.0}
)
print("\n Валідаційні тести пройдено, графіки побудовано!")

#Зображення отриманих метрик та данних у вигляді таблиць

In [ ]:
from IPython.display import display
# таблиця описової статистики вибірки
df_table1 = build_table_1_demographics(df)
# Виводимо без індексів, з вирівнюванням тексту по лівому краю для зручності читання
display(df_table1.style.hide(axis="index").set_properties(**{'text-align': 'left'}))

# таблиця метрик
print("\n\n\033[4m ЗВЕДЕНА ТАБЛИЦЯ РЕЗУЛЬТАТІВ МЕТРИК \033[0m\n")

times_to_evaluate = np.array([24, 60, 120])
min_time = np.percentile(df_test['time'], 10)
max_time = np.percentile(df_test['time'], 90)
brier_times = np.linspace(min_time, max_time, 100)

df_metrics = build_table_2_metrics(
    models_cox=models_cox,
    models_fg_python=models_fg, # словник FG
    model_A_hist=model_A_hist,
    model_B_hist=model_B_hist,
    df_train=df_train,
    df_test=df_test,
    X_test_scaled=X_test_scaled,
    X_test_tensor=X_test_tensor,
    bins=bins,
    brier_times=brier_times,
    times_to_evaluate=times_to_evaluate
)
display(df_metrics.style.set_properties(**{'text-align': 'center'}))


# таблиця гіперпараметрів
print("\n\n\033[4m ТАБЛИЦЯ ГІПЕРПАРАМЕТРІВ \033[0m\n")

df_hyperparams = build_table_3_hyperparams(
    model_instance=model_A_hist,
    batch_size=64,
    lr=0.001,
    num_bins=NUM_BINS,
    patience=15
)
display(df_hyperparams.style.hide(axis="index").set_properties(**{'text-align': 'left'}))